# Multilingual Agreement Analysis

## AA-01. Objective

This notebook evaluates annotation quality and agreement across the
English, Chinese, and Korean sentiment datasets.

The analysis consolidates the final QA results from the three languages
to assess annotation consistency, identify language-specific and
cross-lingual error patterns, and evaluate the effectiveness of the
multilingual annotation guidelines.

### Key Objectives

- Compare annotation quality across English, Chinese, and Korean datasets.
- Evaluate agreement for **Polarity** and **Intensity** labels.
- Identify common and language-specific disagreement patterns.
- Analyze difficult and ambiguous annotation cases.
- Assess the effectiveness of the multilingual annotation guidelines.
- Derive final quality metrics and portfolio-level findings.

## AA-02. Import Libraries

Import the libraries required for multilingual agreement analysis,
quality metric calculation, and visualization.

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    cohen_kappa_score
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## AA-03. Load Final QA Results

Load the final QA-reviewed annotation files for English, Chinese, and Korean datasets.

These datasets will be used for cross-language agreement analysis and final quality evaluation.

In [41]:
# Load final annotation files for each language

df_en = pd.read_csv(
    "../data/annotation/amazon_reviews_en_annotation_300.csv"
)

df_zh = pd.read_csv(
    "../data/annotation/amazon_reviews_zh_annotation_300.csv"
)

df_ko = pd.read_csv(
    "../data/annotation/nsmc_ko_annotation_300.csv"
)

print("English:", df_en.shape)
print("Chinese:", df_zh.shape)
print("Korean:", df_ko.shape)

display(df_en.head())
display(df_zh.head())
display(df_ko.head())

English: (300, 8)
Chinese: (300, 8)
Korean: (300, 8)


,review_id,review_title,review_body,language,product_category,sentiment_label,confidence,notes
0,en_0331970,Good cord,Quick delivery works as it should. Thank you,en,other,NaN,NaN,NaN
1,en_0897981,Perfect lights.,"We have these on the driveway, garage and on the fence in the back yard. No need to leave the ba...",en,home_improvement,NaN,NaN,NaN
2,en_0057491,Great battery life.,"I have many ear buds, the Peats sound great. The battery lasts all day. I have not tried Alex ye...",en,wireless,NaN,NaN,NaN
3,en_0711669,no. just no.,"The other reviews warned about the manual. they were right, it basically just tells you that it'...",en,other,NaN,NaN,NaN
4,en_0873342,good blades,"Watch the online video on how to replace blades. Once I did, it was easy and fast.",en,automotive,NaN,NaN,NaN


,review_id,review_title,review_body,language,product_category,sentiment_label,confidence,notes
0,zh_0958913,感觉还行,某些细节处有线头，布的材质较软，不是想象中那种较硬的帆布，所以实物没有图片看起来那么板正，另外没有味道是真的。背部的拉链和内部是连通的，不像某些仿品还有夹层的，这点感觉仿品做的更实用些，直接连...,zh,shoes,Negative,High,NaN
1,zh_0133459,还不错,平时穿运动鞋42的，皮鞋41的，clarks鞋子一直穿8.5，这双选了8.5刚好。不像评论里过大或者过小。新鞋子，有点硬，穿一穿应该就好了。,zh,shoes,Positive,High,NaN
2,zh_0971712,学mongodb不建议买,书质量很好，但是内容小错不断，有时候容易迷惑，如果是初学者会很费劲,zh,book,Positive,High,NaN
3,zh_0135694,好小,没注意量，以为和超市卖的茄汁罐头一样大，结果好小，太迷你了，虽然挂着进口2字，但比超市大分量的还贵一倍。,zh,grocery,Negative,High,NaN
4,zh_0884672,设计美观、简洁，内容完整无删节,好好的书，送到手时书皮都折了，书角还开裂了！好心疼呜呜呜，只能赶紧用透明胶封一下。除此之外一切都很好，书的大小、字体大小和纸张手感都比较满意，封面也好看。书的内容是未经删改、完整的原版，唯一不...,zh,book,Positive,High,NaN


,review_id,review_title,review_body,language,product_category,sentiment_label,confidence,notes
0,8363132,NaN,진짜 괜찮은데.. 휴 가슴이 답답해질정도로 많은 생각하게 되네요,ko,movie,NaN,NaN,NaN
1,9473399,NaN,BLACK ... 불가능이란건없네요,ko,movie,NaN,NaN,NaN
2,5030353,NaN,갠적으론 8점정도.. 평점이 넘낮아 10점.,ko,movie,NaN,NaN,NaN
3,315189,NaN,호러다큐멘터리라고 잔뜩기대했지만. 호러가 아니었다;,ko,movie,NaN,NaN,NaN
4,4910465,NaN,원작 만화의 원래 제목이 나오코임. 한국에선 스타트로 출간.,ko,movie,NaN,NaN,NaN


## AA-04. Validate Dataset Structure and Label Completeness

Validate the schema, language identifiers, and annotation completeness
before conducting multilingual agreement analysis.

This step checks whether the three datasets share a consistent structure
and whether the final sentiment and confidence labels are available.

In [42]:
# Compare dataset schemas and annotation completeness

datasets = {
    "English": df_en,
    "Chinese": df_zh,
    "Korean": df_ko
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Language values:", df["language"].unique())
    
    print("\nMissing values:")
    print(
        df[
            ["sentiment_label", "confidence", "notes"]
        ].isna().sum()
    )


English
----------------------------------------
Shape: (300, 8)
Columns: ['review_id', 'review_title', 'review_body', 'language', 'product_category', 'sentiment_label', 'confidence', 'notes']
Language values: <ArrowStringArray>
['en']
Length: 1, dtype: str

Missing values:
sentiment_label    300
confidence         300
notes              300
dtype: int64

Chinese
----------------------------------------
Shape: (300, 8)
Columns: ['review_id', 'review_title', 'review_body', 'language', 'product_category', 'sentiment_label', 'confidence', 'notes']
Language values: <ArrowStringArray>
['zh']
Length: 1, dtype: str

Missing values:
sentiment_label     20
confidence          20
notes              300
dtype: int64

Korean
----------------------------------------
Shape: (300, 8)
Columns: ['review_id', 'review_title', 'review_body', 'language', 'product_category', 'sentiment_label', 'confidence', 'notes']
Language values: <ArrowStringArray>
['ko']
Length: 1, dtype: str

Missing values:
sentiment

In [43]:
for name, df in datasets.items():
    print(f"\n{name} sentiment labels:")
    print(df["sentiment_label"].value_counts(dropna=False))

    print(f"\n{name} confidence:")
    print(df["confidence"].value_counts(dropna=False))


English sentiment labels:
sentiment_label
NaN    300
Name: count, dtype: int64

English confidence:
confidence
NaN    300
Name: count, dtype: int64

Chinese sentiment labels:
sentiment_label
Negative    174
Positive    106
NaN          20
Name: count, dtype: int64

Chinese confidence:
confidence
High      154
Medium    102
Low        24
NaN        20
Name: count, dtype: int64

Korean sentiment labels:
sentiment_label
NaN    300
Name: count, dtype: int64

Korean confidence:
confidence
NaN    300
Name: count, dtype: int64


In [44]:
from pathlib import Path

processed_dir = Path("../data/processed")

processed_files = sorted(processed_dir.glob("*.csv"))

for file in processed_files:
    print(file.name)

annotation_en_batch_01.csv
annotation_en_batch_02.csv
annotation_en_batch_03.csv
annotation_en_batch_04.csv
annotation_ko_batch_01.csv
annotation_ko_batch_02.csv
annotation_ko_batch_03.csv
annotation_ko_batch_04.csv
annotation_ko_batch_05.csv
annotation_ko_batch_06.csv
annotation_ko_batch_07.csv
annotation_ko_batch_08.csv
annotation_ko_batch_09.csv
annotation_ko_batch_10.csv
annotation_ko_master.csv
multilingual_qa_portfolio_metrics.csv
qa_disagreements_ko.csv
qa_en_batch_01.csv
qa_en_batch_02.csv
qa_en_batch_03.csv
qa_en_batch_04.csv
qa_metrics_ko.csv
qa_unclear_summary_ko.csv
qa_zh_interim_30_109.csv
qa_zh_interim_summary.csv


In [45]:
# Inspect English annotation and QA file structures

en_annotation_sample = pd.read_csv(
    "../data/processed/annotation_en_batch_01.csv"
)

en_qa_sample = pd.read_csv(
    "../data/processed/qa_en_batch_01.csv"
)

print("=== English Annotation ===")
print("Shape:", en_annotation_sample.shape)
print("Columns:")
print(en_annotation_sample.columns.tolist())

display(en_annotation_sample.head())


print("\n=== English QA ===")
print("Shape:", en_qa_sample.shape)
print("Columns:")
print(en_qa_sample.columns.tolist())

display(en_qa_sample.head())

=== English Annotation ===
Shape: (20, 6)
Columns:
['review_id', 'review_title', 'review_body', 'final_polarity', 'final_intensity', 'language']


,review_id,review_title,review_body,final_polarity,final_intensity,language
0,en_0328488,Unhappy with purchase,Seal was broken. Product looks different than prior purchase.,Negative,Medium,en
1,en_0462975,Don’t purchase,Product didn’t work as stated. Followed directions and it’s currently the 8th day and haven’t se...,Negative,High,en
2,en_0565881,Spilled in shipping,The seeds were received quickly but were spilled out into shipping envelope.,Negative,Low,en
3,en_0636834,Terrible design,This design causes the fan to constantly move away from you and swirl around. Buy one that is en...,Negative,High,en
4,en_0106633,One Star,Bought these and have had them up two weeks and most of them already not working,Negative,Medium,en



=== English QA ===
Shape: (20, 9)
Columns:
['review_id', 'annotator_polarity', 'annotator_intensity', 'qa_polarity', 'qa_intensity', 'polarity_agreement', 'intensity_agreement', 'final_polarity', 'final_intensity']


,review_id,annotator_polarity,annotator_intensity,qa_polarity,qa_intensity,polarity_agreement,intensity_agreement,final_polarity,final_intensity
0,en_0328488,Negative,High,Negative,Medium,True,False,Negative,Medium
1,en_0462975,Negative,High,Negative,High,True,True,Negative,High
2,en_0565881,Negative,High,Negative,Low,True,False,Negative,Low
3,en_0636834,Negative,High,Negative,High,True,True,Negative,High
4,en_0106633,Negative,High,Negative,Medium,True,False,Negative,Medium


In [46]:
# Inspect Chinese and Korean QA file structures

zh_qa_sample = pd.read_csv(
    "../data/processed/qa_zh_interim_30_109.csv"
)

ko_qa_sample = pd.read_csv(
    "../data/processed/qa_disagreements_ko.csv"
)

print("=== Chinese QA ===")
print("Shape:", zh_qa_sample.shape)
print("Columns:")
print(zh_qa_sample.columns.tolist())

display(zh_qa_sample.head())


print("\n=== Korean QA Disagreements ===")
print("Shape:", ko_qa_sample.shape)
print("Columns:")
print(ko_qa_sample.columns.tolist())

display(ko_qa_sample.head())

=== Chinese QA ===
Shape: (21, 8)
Columns:
['review_index', 'initial_sentiment', 'initial_intensity', 'final_sentiment', 'final_intensity', 'sentiment_changed', 'intensity_changed', 'any_changed']


,review_index,initial_sentiment,initial_intensity,final_sentiment,final_intensity,sentiment_changed,intensity_changed,any_changed
0,31,Negative,Low,Negative,Medium,False,True,True
1,34,Positive,Low,Positive,Medium,False,True,True
2,41,Negative,Medium,Negative,High,False,True,True
3,42,Negative,Medium,Positive,Medium,True,False,True
4,47,Positive,High,Positive,Medium,False,True,True



=== Korean QA Disagreements ===
Shape: (40, 8)
Columns:
['id', 'document', 'ground_truth', 'polarity', 'intensity', 'disagreement_type', 'error_category', 'review_note']


,id,document,ground_truth,polarity,intensity,disagreement_type,error_category,review_note
0,8609521,와우!!이건 봐야해!!,Negative,Positive,High,Negative → Positive,Possible Ground-truth Mismatch,Explicit positive recommendation conflicts with negative ground truth.
1,6168434,나라면 귀신한테 복수한다. 꼭,Negative,Unclear,Low,Negative → Unclear,NaN,NaN
2,7610736,이영활 아침 10시에해주길래 우연히 티비틀다보게됐는데 여자애가 어딘가낯이 익긴한데 설마 밀라요보비치겠어?했는데 역시나 밀라였다니 모델로 데뷔한줄 알았는데 영화토박이였구나.....,Negative,Positive,Medium,Negative → Positive,Target Ambiguity,"Strong praise targets the actress, while another character is criticized."
3,8661341,물타기 한번 가볼 까나요...ㅎㅎㅎ,Negative,Unclear,Low,Negative → Unclear,NaN,NaN
4,8083246,걍 레즈비언 영화네...,Negative,Unclear,Low,Negative → Unclear,NaN,NaN


## AA-06. Inspect QA Summary Metrics

Inspect the available QA summary files to determine the total reviewed
sample size and comparable quality metrics for each language.

Because the QA outputs were stored in different formats across languages,
summary-level metrics must be validated before cross-language comparison.

In [47]:
# Inspect available QA summary / metrics files

ko_metrics = pd.read_csv(
    "../data/processed/qa_metrics_ko.csv"
)

zh_summary = pd.read_csv(
    "../data/processed/qa_zh_interim_summary.csv"
)

print("=== Korean QA Metrics ===")
print("Shape:", ko_metrics.shape)
print("Columns:", ko_metrics.columns.tolist())
display(ko_metrics)


print("\n=== Chinese QA Summary ===")
print("Shape:", zh_summary.shape)
print("Columns:", zh_summary.columns.tolist())
display(zh_summary)

=== Korean QA Metrics ===
Shape: (9, 2)
Columns: ['Metric', 'Value']


,Metric,Value
0,Total Samples,300
1,Agreements,260
2,Disagreements,40
3,Overall Agreement Rate,86.7%
4,Decisive Samples,268
5,Decisive Agreements,260
6,Decisive Agreement Rate,97.0%
7,Unclear Disagreements,32
8,Polarity Reversals,8



=== Chinese QA Summary ===
Shape: (5, 2)
Columns: ['Metric', 'Count']


,Metric,Count
0,Total Reviews,80
1,Exact Matches,59
2,Any Changes,21
3,Sentiment Changes,4
4,Intensity Changes,19


## AA-07. Confirm QA Coverage

Confirm the number of samples reviewed during QA for each language
before calculating multilingual agreement metrics.

This prevents interim or disagreement-only records from being
misinterpreted as complete QA results.

In [48]:
# Confirm English QA coverage

en_qa_files = sorted(
    Path("../data/processed").glob("qa_en_batch_*.csv")
)

df_en_qa = pd.concat(
    [pd.read_csv(file) for file in en_qa_files],
    ignore_index=True
)

print("English QA files:", len(en_qa_files))
print("English QA samples:", len(df_en_qa))

print("\nPolarity agreement:")
print(df_en_qa["polarity_agreement"].value_counts(dropna=False))

print("\nIntensity agreement:")
print(df_en_qa["intensity_agreement"].value_counts(dropna=False))

English QA files: 4
English QA samples: 80

Polarity agreement:
polarity_agreement
True     74
False     6
Name: count, dtype: int64

Intensity agreement:
intensity_agreement
True     59
False    21
Name: count, dtype: int64


## AA-08. Multilingual QA Coverage Summary

Summarize annotation volume and QA review coverage across the three
languages before comparing agreement performance.

The project contains 900 annotated samples in total, with different QA
review strategies applied by language. English and Chinese used sampled
QA review, while Korean underwent full-dataset QA review.

Separating annotation volume from QA-reviewed volume ensures that
cross-language quality metrics are interpreted using the correct
evaluation scope.

In [49]:
# Create multilingual QA coverage summary

qa_coverage = pd.DataFrame({
    "Language": ["English", "Chinese", "Korean"],
    "Annotated_Samples": [300, 300, 300],
    "QA_Reviewed_Samples": [80, 80, 300]
})

qa_coverage["QA_Coverage_Rate"] = (
    qa_coverage["QA_Reviewed_Samples"]
    / qa_coverage["Annotated_Samples"]
    * 100
).round(1)

qa_coverage

,Language,Annotated_Samples,QA_Reviewed_Samples,QA_Coverage_Rate
0,English,300,80,26.7
1,Chinese,300,80,26.7
2,Korean,300,300,100.0


In [50]:
# Calculate overall annotation and QA coverage

total_annotated = qa_coverage["Annotated_Samples"].sum()
total_qa_reviewed = qa_coverage["QA_Reviewed_Samples"].sum()

overall_qa_coverage = (
    total_qa_reviewed / total_annotated * 100
)

print(f"Total annotated samples: {total_annotated}")
print(f"Total QA-reviewed samples: {total_qa_reviewed}")
print(f"Overall QA coverage: {overall_qa_coverage:.1f}%")

Total annotated samples: 900
Total QA-reviewed samples: 460
Overall QA coverage: 51.1%


### Interpretation

- A total of **900 multilingual samples** were annotated across English,
  Chinese, and Korean.
- **460 samples** were subsequently evaluated through QA, representing
  **51.1% of the full annotation dataset**.
- English and Chinese each used an **80-sample QA review (26.7%)**,
  while Korean received **full-dataset QA coverage (100%)**.
- Because QA coverage differs by language, subsequent agreement metrics
  are compared using the reviewed sample population rather than the
  full annotation volume.

## AA-09. Polarity Agreement Analysis

Compare polarity agreement across English, Chinese, and Korean QA results.

Polarity agreement measures whether the initial annotation and QA review
reached the same sentiment decision.

For Korean, both overall agreement and decisive agreement are reported
because the dataset includes `Unclear` cases that require separate
interpretation.

In [51]:
# English polarity agreement

en_polarity_total = len(df_en_qa)
en_polarity_agree = df_en_qa["polarity_agreement"].sum()

en_polarity_rate = (
    en_polarity_agree / en_polarity_total * 100
)

print("English Polarity QA")
print("-" * 30)
print("Reviewed:", en_polarity_total)
print("Agreements:", en_polarity_agree)
print("Disagreements:", en_polarity_total - en_polarity_agree)
print(f"Agreement Rate: {en_polarity_rate:.1f}%")

English Polarity QA
------------------------------
Reviewed: 80
Agreements: 74
Disagreements: 6
Agreement Rate: 92.5%


In [52]:
# Chinese polarity agreement

zh_total = 80
zh_polarity_changes = 4
zh_polarity_agree = zh_total - zh_polarity_changes

zh_polarity_rate = (
    zh_polarity_agree / zh_total * 100
)

print("Chinese Polarity QA")
print("-" * 30)
print("Reviewed:", zh_total)
print("Agreements:", zh_polarity_agree)
print("Disagreements:", zh_polarity_changes)
print(f"Agreement Rate: {zh_polarity_rate:.1f}%")

Chinese Polarity QA
------------------------------
Reviewed: 80
Agreements: 76
Disagreements: 4
Agreement Rate: 95.0%


In [53]:
# Multilingual polarity agreement summary

polarity_summary = pd.DataFrame({
    "Language": ["English", "Chinese", "Korean"],
    "QA_Reviewed": [80, 80, 300],
    "Agreements": [74, 76, 260],
    "Disagreements": [6, 4, 40],
    "Overall_Agreement_Rate": [92.5, 95.0, 86.7]
})

polarity_summary

,Language,QA_Reviewed,Agreements,Disagreements,Overall_Agreement_Rate
0,English,80,74,6,92.5
1,Chinese,80,76,4,95.0
2,Korean,300,260,40,86.7


In [54]:
ko_decisive_rate = 260 / 268 * 100

print(f"Korean overall agreement: {260 / 300 * 100:.1f}%")
print(f"Korean decisive agreement: {ko_decisive_rate:.1f}%")
print("Korean unclear disagreements: 32")

Korean overall agreement: 86.7%
Korean decisive agreement: 97.0%
Korean unclear disagreements: 32


### Interpretation

- **Chinese achieved the highest polarity agreement (95.0%)**, followed
  by English (92.5%).
- Korean showed a lower overall agreement rate (86.7%), largely because
  `Unclear` cases were included in the evaluation.
- When only decisive Korean samples were considered, agreement increased
  to **97.0%**, indicating strong consistency for clearly classifiable
  sentiment.
- Across all three languages, polarity classification was relatively
  stable, while ambiguous cases had a greater impact on Korean overall
  agreement.

## AA-10. Intensity Agreement Analysis

Evaluate agreement for sentiment intensity labels and compare the results
with polarity agreement.

Intensity classification distinguishes the strength of sentiment using
`Low`, `Medium`, and `High` labels.

Comparable row-level intensity QA results are available for English and
Chinese. Korean intensity agreement is not included in this direct
comparison because an equivalent final intensity QA metric was not
stored in the Korean QA output.

In [55]:
# English intensity agreement

en_intensity_total = len(df_en_qa)
en_intensity_agree = df_en_qa["intensity_agreement"].sum()
en_intensity_disagree = en_intensity_total - en_intensity_agree

en_intensity_rate = (
    en_intensity_agree / en_intensity_total * 100
)

print("English Intensity QA")
print("-" * 30)
print("Reviewed:", en_intensity_total)
print("Agreements:", en_intensity_agree)
print("Disagreements:", en_intensity_disagree)
print(f"Agreement Rate: {en_intensity_rate:.2f}%")

English Intensity QA
------------------------------
Reviewed: 80
Agreements: 59
Disagreements: 21
Agreement Rate: 73.75%


In [56]:
# Chinese intensity agreement

zh_intensity_total = 80
zh_intensity_disagree = 19
zh_intensity_agree = zh_intensity_total - zh_intensity_disagree

zh_intensity_rate = (
    zh_intensity_agree / zh_intensity_total * 100
)

print("Chinese Intensity QA")
print("-" * 30)
print("Reviewed:", zh_intensity_total)
print("Agreements:", zh_intensity_agree)
print("Disagreements:", zh_intensity_disagree)
print(f"Agreement Rate: {zh_intensity_rate:.2f}%")

Chinese Intensity QA
------------------------------
Reviewed: 80
Agreements: 61
Disagreements: 19
Agreement Rate: 76.25%


In [57]:
# Compare polarity and intensity agreement

dimension_comparison = pd.DataFrame({
    "Language": ["English", "Chinese"],
    "Polarity_Agreement": [
        en_polarity_rate,
        zh_polarity_rate
    ],
    "Intensity_Agreement": [
        en_intensity_rate,
        zh_intensity_rate
    ]
})

dimension_comparison["Agreement_Gap"] = (
    dimension_comparison["Polarity_Agreement"]
    - dimension_comparison["Intensity_Agreement"]
).round(2)

dimension_comparison.round(2)

,Language,Polarity_Agreement,Intensity_Agreement,Agreement_Gap
0,English,92.5,73.75,18.75
1,Chinese,95.0,76.25,18.75


### Interpretation

- Intensity agreement was substantially lower than polarity agreement
  in both English and Chinese.
- English achieved **92.5% polarity agreement** but only **73.75%
  intensity agreement**, a gap of **18.75 percentage points**.
- Chinese showed the same pattern, with **95.0% polarity agreement**
  versus **76.25% intensity agreement**, also an **18.75-point gap**.
- The consistent gap across both languages suggests that determining
  sentiment strength is more subjective than identifying sentiment
  direction.
- This finding indicates that intensity definitions and boundary cases
  should receive greater emphasis in future annotation guidelines and
  calibration examples.

## AA-11. Cohen's Kappa Analysis

Cohen's Kappa is used to evaluate agreement beyond simple percentage
agreement by accounting for agreement that could occur by chance.

Because complete paired annotator and QA labels are available for the
English QA sample, Cohen's Kappa is calculated for English polarity and
intensity annotations.

Kappa is not calculated for Chinese or Korean because equivalent
row-level paired QA records were not preserved in the final QA outputs.
This avoids making cross-language comparisons using non-equivalent data.

In [58]:
# Calculate Cohen's Kappa for English polarity

en_polarity_kappa = cohen_kappa_score(
    df_en_qa["annotator_polarity"],
    df_en_qa["qa_polarity"]
)

print(f"English Polarity Cohen's Kappa: {en_polarity_kappa:.3f}")

English Polarity Cohen's Kappa: 0.000


In [59]:
# Calculate Cohen's Kappa for English intensity

en_intensity_kappa = cohen_kappa_score(
    df_en_qa["annotator_intensity"],
    df_en_qa["qa_intensity"]
)

print(f"English Intensity Cohen's Kappa: {en_intensity_kappa:.3f}")

English Intensity Cohen's Kappa: 0.495


In [60]:
# Summarize English agreement metrics

en_kappa_summary = pd.DataFrame({
    "Dimension": ["Polarity", "Intensity"],
    "Percent_Agreement": [
        en_polarity_rate,
        en_intensity_rate
    ],
    "Cohens_Kappa": [
        en_polarity_kappa,
        en_intensity_kappa
    ]
})

en_kappa_summary.round(3)

,Dimension,Percent_Agreement,Cohens_Kappa
0,Polarity,92.50,0.000
1,Intensity,73.75,0.495


In [61]:
# Inspect English polarity label distribution

print("=== Annotator Polarity ===")
print(
    df_en_qa["annotator_polarity"]
    .value_counts(dropna=False)
)

print("\n=== QA Polarity ===")
print(
    df_en_qa["qa_polarity"]
    .value_counts(dropna=False)
)

print("\n=== Polarity Crosstab ===")
display(
    pd.crosstab(
        df_en_qa["annotator_polarity"],
        df_en_qa["qa_polarity"],
        margins=True
    )
)

=== Annotator Polarity ===
annotator_polarity
Negative    74
Mixed        3
Neutral      2
Unclear      1
Name: count, dtype: int64

=== QA Polarity ===
qa_polarity
Negative    80
Name: count, dtype: int64

=== Polarity Crosstab ===


qa_polarity,Negative,All
annotator_polarity,,
Mixed,3,3
Negative,74,74
Neutral,2,2
Unclear,1,1
All,80,80


In [62]:
# Inspect English intensity label distribution

print("=== Annotator Intensity ===")
print(
    df_en_qa["annotator_intensity"]
    .value_counts(dropna=False)
)

print("\n=== QA Intensity ===")
print(
    df_en_qa["qa_intensity"]
    .value_counts(dropna=False)
)

print("\n=== Intensity Crosstab ===")
display(
    pd.crosstab(
        df_en_qa["annotator_intensity"],
        df_en_qa["qa_intensity"],
        margins=True
    )
)

=== Annotator Intensity ===
annotator_intensity
Medium    45
High      31
Low        4
Name: count, dtype: int64

=== QA Intensity ===
qa_intensity
Medium    50
High      26
Low        4
Name: count, dtype: int64

=== Intensity Crosstab ===


qa_intensity,High,Low,Medium,All
annotator_intensity,,,,
High,18,1,12,31
Low,1,3,0,4
Medium,7,0,38,45
All,26,4,50,80


### Interpretation

- English polarity showed a high raw agreement rate of **92.5%**
  (74/80).
- However, Cohen's Kappa for polarity was **0.000** because the QA
  reviewer assigned all 80 samples to the `Negative` category.
- This single-class QA distribution makes polarity Kappa unsuitable
  as a standalone measure of annotation quality in this sample.
- Therefore, polarity performance is reported primarily using percent
  agreement together with the observed label distribution.
- For intensity, both the annotator and QA reviewer used all three
  categories (`Low`, `Medium`, and `High`), allowing a more meaningful
  chance-corrected comparison.
- Intensity achieved **73.75% raw agreement** and a Cohen's Kappa of
  **0.495**, indicating moderate agreement beyond chance.
- These results demonstrate the importance of examining label
  distributions alongside aggregate agreement metrics rather than
  interpreting a single metric in isolation.

## AA-12. Error Pattern Analysis

Analyze disagreement patterns to identify the main sources of annotation
difficulty across languages.

Rather than relying only on aggregate agreement rates, this section
examines which sentiment dimensions and ambiguity types contributed to
QA disagreements.

Because QA records were stored differently across languages, error
patterns are first analyzed within each language before deriving
cross-language findings.

In [63]:
# Analyze English disagreement patterns

en_error_summary = pd.DataFrame({
    "Error_Type": [
        "Polarity Disagreement",
        "Intensity Disagreement",
        "Any Disagreement"
    ],
    "Count": [
        (~df_en_qa["polarity_agreement"]).sum(),
        (~df_en_qa["intensity_agreement"]).sum(),
        (
            (~df_en_qa["polarity_agreement"])
            | (~df_en_qa["intensity_agreement"])
        ).sum()
    ]
})

en_error_summary

,Error_Type,Count
0,Polarity Disagreement,6
1,Intensity Disagreement,21
2,Any Disagreement,26


In [64]:
# Analyze Chinese disagreement patterns

zh_error_summary = pd.DataFrame({
    "Error_Type": [
        "Sentiment Change",
        "Intensity Change",
        "Any Change"
    ],
    "Count": [
        zh_qa_sample["sentiment_changed"].sum(),
        zh_qa_sample["intensity_changed"].sum(),
        zh_qa_sample["any_changed"].sum()
    ]
})

zh_error_summary

,Error_Type,Count
0,Sentiment Change,4
1,Intensity Change,19
2,Any Change,21


In [65]:
# Analyze Korean disagreement patterns

print("=== Korean Disagreement Types ===")

ko_disagreement_types = (
    ko_qa_sample["disagreement_type"]
    .value_counts(dropna=False)
    .rename_axis("Disagreement_Type")
    .reset_index(name="Count")
)

display(ko_disagreement_types)


print("\n=== Korean Error Categories ===")

ko_error_categories = (
    ko_qa_sample["error_category"]
    .value_counts(dropna=False)
    .rename_axis("Error_Category")
    .reset_index(name="Count")
)

display(ko_error_categories)

=== Korean Disagreement Types ===


,Disagreement_Type,Count
0,Positive → Unclear,19
1,Negative → Unclear,13
2,Negative → Positive,4
3,Positive → Negative,4



=== Korean Error Categories ===


,Error_Category,Count
0,NaN,32
1,Possible Ground-truth Mismatch,3
2,Mixed Sentiment,3
3,Target Ambiguity,1
4,Implicit / Context-dependent,1


In [66]:
# Analyze polarity decisions among Korean disagreement cases

ko_disagreement_polarity = (
    ko_qa_sample["polarity"]
    .value_counts(dropna=False)
    .rename_axis("Polarity")
    .reset_index(name="Count")
)

display(ko_disagreement_polarity)

,Polarity,Count
0,Unclear,32
1,Positive,4
2,Negative,4


### Interpretation

- English recorded **26 samples with at least one disagreement**, while
  Chinese recorded **21 samples with at least one QA change**.
- In both English and Chinese, intensity-related disagreements were much
  more frequent than polarity-related disagreements, reinforcing
  intensity classification as a common annotation challenge.
- Korean produced **40 polarity disagreements**, of which **32 (80.0%)**
  involved a transition from an initial Positive or Negative label to
  `Unclear`.
- Only **8 Korean cases (20.0%)** represented direct Positive–Negative
  polarity reversals.
- Among the explicitly categorized Korean reversal cases, the main
  issues included **possible ground-truth mismatch**, **mixed sentiment**,
  **target ambiguity**, and **implicit/context-dependent sentiment**.
- Overall, disagreement was driven less by simple sentiment-direction
  errors and more by **sentiment strength, ambiguity, mixed signals,
  and contextual interpretation**.

## AA-13. Cross-Language Error Findings

Cross-language comparison reveals that annotation difficulty was not
primarily caused by basic Positive–Negative sentiment classification.

Instead, disagreement concentrated around more subjective dimensions
such as sentiment strength, ambiguous expressions, mixed sentiment,
and context-dependent interpretation.

### Common Patterns

1. **Polarity was relatively stable**
   - English polarity agreement: **92.5%**
   - Chinese polarity agreement: **95.0%**
   - Korean decisive polarity agreement: **97.0%**

2. **Intensity was a major cross-language challenge**
   - English intensity agreement: **73.75%**
   - Chinese intensity agreement: **76.25%**
   - Both languages showed an **18.75 percentage-point gap** between
     polarity and intensity agreement.

3. **Ambiguity strongly affected Korean QA**
   - **32 of 40 Korean disagreements (80.0%)** were resolved as `Unclear`.
   - Direct Positive–Negative reversals accounted for only **8 cases**.

4. **Complex sentiment structures created difficult edge cases**
   - Mixed sentiment
   - Target ambiguity
   - Implicit or context-dependent sentiment
   - Possible ground-truth mismatch

### Cross-Language Finding

The results suggest that multilingual sentiment annotation becomes more
difficult as the task moves from identifying basic sentiment direction
toward interpreting **strength, ambiguity, mixed signals, and contextual
meaning**.

## AA-14. Guideline Effectiveness & Improvement

Evaluate the annotation guideline based on the disagreement patterns
identified during multilingual QA.

The QA results indicate that basic sentiment direction was generally
consistent across languages, while intensity boundaries, ambiguous
expressions, mixed sentiment, and context-dependent cases required
greater interpretive judgment.

These findings provide evidence for refining the guideline with clearer
decision rules and additional edge-case examples.

### Guideline Strengths

- Positive and Negative polarity definitions produced relatively high
  agreement across all three languages.
- The guideline provided a consistent framework for multilingual
  sentiment classification.
- QA review successfully identified ambiguous cases that should not be
  forced into a binary sentiment decision.
- The `Unclear` label was particularly useful for separating genuinely
  ambiguous Korean cases from direct polarity errors.

### Areas for Improvement

#### 1. Intensity Boundaries

English and Chinese showed substantially lower agreement for intensity
than for polarity.

Future guidelines should provide clearer boundaries between:

- `Low` — weak or mildly expressed sentiment
- `Medium` — clear sentiment without strong emphasis
- `High` — explicit, strongly emphasized, or emotionally intense sentiment

Additional borderline examples should be provided for each level.

#### 2. Mixed Sentiment

Reviews containing both positive and negative opinions require a more
explicit decision rule.

The guideline should clarify whether annotators should prioritize:

- the overall evaluation,
- the dominant sentiment,
- the final recommendation, or
- `Unclear` when no dominant sentiment can be established.

#### 3. Target Ambiguity

Sentiment toward the product, delivery, seller, service, or another
entity may appear in the same review.

The guideline should explicitly define the primary sentiment target and
provide examples of conflicting targets.

#### 4. Implicit and Context-Dependent Sentiment

Sarcasm, indirect criticism, culturally dependent expressions, and
context-dependent language may not contain explicit sentiment words.

These cases should be supported with language-specific examples and
escalated to `Unclear` when evidence is insufficient.

#### 5. QA Calibration

Future annotation rounds should include a small calibration batch before
full-scale labeling.

Disagreements from the calibration round can be reviewed to refine
definitions and examples before annotators proceed with the remaining
dataset.

### Recommended Guideline Updates

| QA Finding | Guideline Update |
|---|---|
| Lower intensity agreement | Add explicit Low / Medium / High boundary rules and borderline examples |
| Mixed sentiment cases | Define dominant-sentiment and tie-breaking rules |
| Target ambiguity | Specify the primary sentiment target and target-priority rules |
| Context-dependent sentiment | Add language-specific implicit and contextual examples |
| Frequent Unclear cases | Define when `Unclear` should be used versus forced polarity assignment |
| Cross-language variation | Add calibration examples separately for EN, ZH, and KO |

## AA-15. Key Findings

The multilingual agreement analysis identified several key findings
regarding annotation quality, disagreement patterns, and QA effectiveness.

### 1. Multilingual Annotation and QA Coverage

- **900 samples** were annotated across English, Chinese, and Korean.
- **460 samples** were evaluated through QA, representing **51.1%**
  of the full annotation dataset.
- English and Chinese used sampled QA review (80 samples each), while
  Korean received full-dataset QA review (300 samples).

### 2. Polarity Classification Was Highly Consistent

- English polarity agreement: **92.5%**
- Chinese polarity agreement: **95.0%**
- Korean decisive polarity agreement: **97.0%**
- These results indicate that clear Positive–Negative sentiment
  decisions were generally stable across languages.

### 3. Intensity Was the Main Cross-Language QA Challenge

- English intensity agreement: **73.75%**
- Chinese intensity agreement: **76.25%**
- Both languages showed an **18.75 percentage-point gap** between
  polarity and intensity agreement.
- This suggests that sentiment strength is substantially more
  subjective than sentiment direction.

### 4. Ambiguity Drove Most Korean Disagreements

- Korean overall agreement was **86.7%**.
- **32 of 40 disagreements (80.0%)** were resolved as `Unclear`.
- Only **8 cases** represented direct Positive–Negative reversals.
- When `Unclear` cases were excluded, decisive agreement increased
  to **97.0%**.

### 5. Error Analysis Identified Recurring Edge Cases

Major sources of annotation difficulty included:

- intensity boundary ambiguity,
- mixed sentiment,
- target ambiguity,
- implicit or context-dependent sentiment,
- and possible ground-truth mismatch.

These patterns indicate that complex sentiment interpretation, rather
than basic polarity recognition, was the primary source of disagreement.

### 6. QA Findings Led to Actionable Guideline Improvements

The analysis identified concrete opportunities to improve future
annotation rounds through:

- clearer intensity boundary definitions,
- explicit mixed-sentiment decision rules,
- sentiment-target prioritization,
- stronger `Unclear` criteria,
- language-specific edge-case examples,
- and pre-annotation calibration batches.

## AA-16. Multilingual Final Evaluation

The multilingual sentiment annotation project demonstrated generally
strong consistency in polarity classification across English, Chinese,
and Korean, while also revealing important challenges in sentiment
intensity and ambiguous language.

Across the project, **900 samples** were annotated and **460 samples**
were evaluated through QA. Polarity agreement remained high across
languages, with English and Chinese achieving **92.5%** and **95.0%**
agreement respectively, while Korean achieved **97.0% agreement among
decisive cases**.

The most consistent cross-language challenge was sentiment intensity.
English and Chinese intensity agreement decreased to **73.75%** and
**76.25%**, respectively. In both languages, this represented an
**18.75 percentage-point decline** compared with polarity agreement,
indicating that sentiment strength requires more subjective judgment
than sentiment direction.

Korean QA revealed a different but complementary challenge. Although
overall agreement was **86.7%**, 80% of Korean disagreements involved
cases ultimately classified as `Unclear`. Direct Positive–Negative
reversals were relatively limited, suggesting that ambiguity rather
than basic polarity misclassification was the primary source of
disagreement.

Error analysis further identified mixed sentiment, target ambiguity,
implicit or context-dependent expressions, and possible ground-truth
mismatch as recurring edge cases.

Overall, the QA results indicate that the annotation framework was
effective for clear sentiment polarity but requires stronger decision
rules for **intensity boundaries, ambiguity, mixed sentiment, and
context-dependent interpretation**.

These findings provide a practical basis for improving future annotation
guidelines, calibration procedures, and multilingual QA workflows.

### Evaluation Limitations

- QA coverage differed across languages: English and Chinese used
  80-sample QA subsets, while Korean received full 300-sample review.
- The English QA subset was strongly skewed toward Negative sentiment,
  limiting the interpretability of polarity Cohen's Kappa.
- QA outputs were stored using different structures across languages,
  limiting direct comparison of some metrics.
- Equivalent row-level intensity agreement data was not preserved for
  Korean, so the cross-language intensity comparison was limited to
  English and Chinese.
- The current findings should therefore be interpreted as portfolio-level
  QA evidence rather than as a controlled benchmark comparison among
  the three languages.

## AA-17. Portfolio Metrics Summary

Summarize the key quantitative results from the multilingual annotation
and QA workflow for portfolio reporting.

The metrics below distinguish annotation volume from QA-reviewed volume
and report only directly comparable agreement measures.

In [67]:
# Create final portfolio metrics summary

portfolio_metrics = pd.DataFrame({
    "Metric": [
        "Languages",
        "Total Annotated Samples",
        "Total QA-Reviewed Samples",
        "Overall QA Coverage",
        "English Polarity Agreement",
        "Chinese Polarity Agreement",
        "Korean Overall Agreement",
        "Korean Decisive Agreement",
        "English Intensity Agreement",
        "Chinese Intensity Agreement",
        "Korean Unclear Share of Disagreements"
    ],
    "Value": [
        "3 (EN / ZH / KO)",
        "900",
        "460",
        "51.1%",
        "92.5%",
        "95.0%",
        "86.7%",
        "97.0%",
        "73.75%",
        "76.25%",
        "80.0%"
    ]
})

portfolio_metrics

,Metric,Value
0,Languages,3 (EN / ZH / KO)
1,Total Annotated Samples,900
2,Total QA-Reviewed Samples,460
3,Overall QA Coverage,51.1%
4,English Polarity Agreement,92.5%
5,Chinese Polarity Agreement,95.0%
6,Korean Overall Agreement,86.7%
7,Korean Decisive Agreement,97.0%
8,English Intensity Agreement,73.75%
9,Chinese Intensity Agreement,76.25%


### Portfolio Highlight

**Multilingual Sentiment Annotation & QA**

- Annotated **900 samples across 3 languages (EN / ZH / KO)**.
- Conducted QA review on **460 samples (51.1% overall QA coverage)**.
- Achieved **92.5%–97.0% agreement for decisive polarity classification**.
- Identified sentiment **intensity and ambiguity** as the primary
  cross-language QA challenges.
- Detected an **18.75 percentage-point polarity–intensity agreement gap**
  in both English and Chinese QA samples.
- Converted disagreement analysis into actionable guideline improvements
  for intensity boundaries, mixed sentiment, target ambiguity, and
  context-dependent expressions.

In [68]:
# Save portfolio-level metrics

output_path = "../data/processed/multilingual_qa_portfolio_metrics.csv"

portfolio_metrics.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {output_path}")

Saved: ../data/processed/multilingual_qa_portfolio_metrics.csv
